# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*  
**One row** = One page for one client on one day  
**Table(s)**: fact_content_daily_performance and dim_content fact_content_query_90d wasnt picked because it includes the sealed month in its window (april-june 2026)  
**Time window**: month=2026-03  
**Label/proxy**: rank by ctr_gap  
**Exclusion**: raw ctr/clicks/impressions as features (used to build the label)  

In [5]:
import os
from huggingface_hub import login
import duckdb
#note for self: this is the best practice for using tokens for safety, u make it using environment variables by doing this:
#setx NAME_OF_TOKEN "TOKEN ID HERE"

HF_TOKEN = os.environ.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

fact_content_daily_performance_table = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")
dim_content_table  = con.sql(f"SELECT * from read_parquet('{rel}/dim_content.parquet')")

con.sql("SELECT * FROM fact_content_daily_performance_table LIMIT 5").show()
con.sql("SELECT * FROM dim_content_table LIMIT 5").show()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**features**: gsc_avg_position, search_volume, competition_level, main_intent  
**label**: ctr_gap  
**context**: content_hash_id, client_hash_id, report_date, gsc_data_available , is_deleted   
**excluded**: using (gsc_clicks, gsc_impressions) as direct features beacuse they give leakage, and ALL GA4/session columns,  
because they are available AFTER u click on the page, so they are basically the same as gsc_clicks.

In [6]:
#describe is used to give data type of each column 
con.sql(f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM fact_content_daily_performance_table)").show()
con.sql(f"SELECT column_name, column_type FROM (DESCRIBE SELECT * FROM dim_content_table)").show()

#this is how to print the column names as a list, .columns is an attribute not a method thats why its not .columns(). 
print(f"fact_content_daily_performance_table columns: {con.sql(f"SELECT * FROM fact_content_daily_performance_table WHERE 1=0").columns}")
print(f"dim_content_table columns: {con.sql(f"SELECT * FROM dim_content_table WHERE 1=0").columns}")

┌────────────────────┬─────────────┐
│    column_name     │ column_type │
│      varchar       │   varchar   │
├────────────────────┼─────────────┤
│ report_date        │ DATE        │
│ client_hash_id     │ VARCHAR     │
│ content_hash_id    │ VARCHAR     │
│ client_has_gsc     │ BOOLEAN     │
│ client_has_ga4     │ BOOLEAN     │
│ gsc_data_available │ BOOLEAN     │
│ ga4_data_available │ BOOLEAN     │
│ gsc_impressions    │ BIGINT      │
│ gsc_clicks         │ BIGINT      │
│ gsc_sum_position   │ BIGINT      │
│      ·             │   ·         │
│      ·             │   ·         │
│      ·             │   ·         │
│ sessions_ai        │ BIGINT      │
│ ai_chatgpt         │ BIGINT      │
│ ai_perplexity      │ BIGINT      │
│ ai_gemini          │ BIGINT      │
│ ai_copilot         │ BIGINT      │
│ ai_claude          │ BIGINT      │
│ ai_meta            │ BIGINT      │
│ ai_other           │ BIGINT      │
│ scroll_events      │ BIGINT      │
│ month              │ VARCHAR     │
└

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
import pandas as pd
#why using HAVING not WHERE? beacuse WHERE checks before group by finishes, HAVING comes with GROUP BY .
#also the point of this is to prove each combination of (report_date , client_id, content_id) doesnt get repeated. 
#this is what's called "grain", which the section asks for
print("fact_content_daily_performance_table grain: ")
con.sql("""SELECT report_date , client_hash_id, content_hash_id, COUNT(*) AS n FROM fact_content_daily_performance_table
            GROUP BY report_date , client_hash_id, content_hash_id HAVING COUNT(*) > 1""").show()

print("dim_content_table grain: ")
con.sql("SELECT content_hash_id, COUNT(*) AS n FROM dim_content_table GROUP BY content_hash_id HAVING COUNT(*) > 1").show()


print("For fact_content_daily_performance_table: ")

#note: pct stands for percentage.
#using IS TRUE instead of = true is because of NULL exisitng as a value for gsc_data_available, and SQL treats it differently so just incase i used IS TRUE
con.sql("""SELECT COUNT(*) AS pages_count, COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_pages, 
ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 2) AS pct_available
FROM fact_content_daily_performance_table""").show()

#this to prove database is actually just 1 month no other months in it
con.sql("SELECT MIN(report_date) AS earliest_date, MAX(report_date) AS latest_date from fact_content_daily_performance_table").show()

#building the dataframe
df = con.sql("""SELECT f.content_hash_id, f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position, d.search_volume, d.competition_level, d.main_intent
             FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING (content_hash_id)
             WHERE f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND gsc_avg_position > 0""").df()

#pd.cut() works to make each interval have a label, the ranges are 1-10, 11-20, 21+
#float("inf") is python's way of tying positive infinity
df["position_tier"] = pd.cut(df["gsc_avg_position"], bins=[0,10,20,float("inf")], labels=["page_1", "striking", "page_3_5"])

#.mask(cond) works exactly as NULLIF in SQL, when the cond is true, it replaces the value with NaN
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].mask(df["gsc_impressions"] == 0)

#the reason to do allat is get a better mean
expected_ctr_per_tier = df.groupby("position_tier")["gsc_clicks"].sum() / df.groupby("position_tier")["gsc_impressions"].sum()

#how map works: for each row, try to find the position tier in the map, and puts it back in the new column "expected_ctr"
#first of all, .astype(type) forces the type to the column, second of all, the reason i did that was because pd.cut() returns a panda type called category
#and when i did .map(), even though the mapping happens on a float type column, panda makes it a category type
df["expected_ctr"] = df["position_tier"].map(expected_ctr_per_tier).astype(float)

#IF i didn't do astype(float) here, meaning expected_ctr column is a category type, and panda doesnt accept math operations on category type columns, so it would give a typeERROR
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]
df.head(10).sort_values("ctr_gap", ascending=False)\
  .style.format("{:.3f}",subset=["expected_ctr", "ctr", "ctr_gap"]).set_properties(**{"text-align": "center"})

fact_content_daily_performance_table grain: 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
└─────────────┴────────────────┴─────────────────┴───────┘
                          0 rows                        

dim_content_table grain: 
┌─────────────────┬───────┐
│ content_hash_id │   n   │
│     varchar     │ int64 │
└─────────────────┴───────┘
          0 rows         

For fact_content_daily_performance_table: 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬─────────────────┬───────────────┐
│ pages_count │ available_pages │ pct_available │
│    int64    │      int64      │    double     │
├─────────────┼─────────────────┼───────────────┤
│     9841378 │         3611061 │         36.69 │
└─────────────┴─────────────────┴───────────────┘

┌───────────────┬─────────────┐
│ earliest_date │ latest_date │
│     date      │    date     │
├───────────────┼─────────────┤
│ 2026-03-01    │ 2026-03-31  │
└───────────────┴─────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,search_volume,competition_level,main_intent,position_tier,ctr,expected_ctr,ctr_gap
0,content_f76b13bbc974958a,0,8,5.375000,20,HIGH,transactional,page_1,0.000,0.003,0.003
2,content_2ae00e445b2b6a0d,0,38,5.763158,20,LOW,commercial,page_1,0.000,0.003,0.003
8,content_c233b46116244a98,0,1,8.000000,20,LOW,transactional,page_1,0.000,0.003,0.003
5,content_fc3f33f6a4aed41d,0,46,5.717391,30,LOW,transactional,page_1,0.000,0.003,0.003
9,content_1a0cb7648dc42bd1,0,169,1.260355,30,LOW,commercial,page_1,0.000,0.003,0.003
1,content_fc2502eb1fd34754,0,1,96.000000,20,MEDIUM,commercial,page_3_5,0.000,0.001,0.001
4,content_c6f5788b1d82deb1,0,9,33.444444,20,LOW,informational,page_3_5,0.000,0.001,0.001
3,content_6935ec4367fdc955,0,6,71.833333,20,HIGH,transactional,page_3_5,0.000,0.001,0.001
7,content_1ddf9509dfdbef75,1,308,5.983766,20,LOW,transactional,page_1,0.003,0.003,0.000
6,content_6bdb8e786a783b8d,1,9,3.222222,30,LOW,transactional,page_1,0.111,0.003,-0.108


**gsc_avg_position** - knowable at the decision moment because its the page's search ranking, on the same day before any clicks happen.    
**search_volume** - knowable at the decision moment because its a property of the keyword which happens before any clicks.  
**competition_level** - knowable at the decision moment because of the same reason as search_volume.  
**main_intent** - knowable at the decision moment because its from the content itself, not from CTR or any clicks.  

**Leakage trap:** my lane is just ranking not prediction, so there is no future outcome being predicted, for now atleast, so there is technically no leakage here.  

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*  
out of all pages, only about 37% of them have gsc_data, meaning the rest of the 63% are invisible to me and i cant do ranking on them, that is a real data limit.

In [8]:
# Proof for this limitation already shown in Section 3's verification query:
# COUNT(*) AS pages_count, COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_pages
# Result: 3,611,061 available out of 9,841,378 total (~37%)

## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.